In [43]:
import os
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
import json
import sqlite3

In [44]:
load_dotenv(override=True)

google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')

if not google_api_key:
    raise ValueError("GOOGLE_API_KEY environment variable is not set.")
if not groq_api_key:
    raise ValueError("GROQ_API_KEY environment variable is not set.")


In [62]:
MODEL_groq = "llama-3.3-70b-versatile" # groq
MODEL_gemini ="gemini-1.5-flash-latest"  # gemini

In [46]:
openai = OpenAI()

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [63]:
system_message = "You are a helpful assistant that can answer questions about the world and also use tools to get information. You have access to the following tools:"


response = groq.chat.completions.create(
    model=MODEL_groq,
    messages=[{"role": "system", "content": system_message}, {"role": "user", "content": "What is the capital of France?"}]
)   

print (response.choices[0].message.content)

The capital of France is Paris.


In [47]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [48]:

set_price_function = {
    "name": "set_ticket_price",
    "description": "Set the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
            "price": {
                "type": "number",
                "description": "The price to set for the ticket",
            }
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

In [49]:
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": set_price_function}]

In [50]:
DB = "prices1.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [51]:

# ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [65]:
get_ticket_price("Berlin")

DATABASE TOOL CALLED: Getting price for Berlin


'Ticket price to Berlin is $600.0'

In [53]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [54]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1320, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [55]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1320.0'

In [64]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
Always confirm when a price has been successfully updated.
"""

def handle_tool_calls(message):

    responses = []
    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        if function_name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        elif function_name == "set_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price = arguments.get('price')
            set_ticket_price(city, price)
            responses.append({
                "role": "tool",
                "content": f"Price for ticket to {city} set to {price}",
                "tool_call_id": tool_call.id
            })
    return responses

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL_groq, messages=messages, tools=tools)


# doesnt allow more than 1 set of tool call sequentially
    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(model=MODEL_groq, messages=messages)

        for message in messages:
            print(message)
    
    return response.choices[0].message.content


gr.ChatInterface(
    fn=chat,
    type="messages"
).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for London
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\nAlways confirm when a price has been successfully updated.\n"}
{'role': 'user', 'content': 'hi there i want to travel to london. if the ticket price tolondon is less than $1000 then check ticket price to Paris '}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='9g4vwn9y7', function=Function(arguments='{"destination_city":"London"}', name='get_ticket_price'), type='function')])
{'role': 'tool', 'content': 'Ticket price to London is $799.0', 'tool_call_id': '9g4vwn9y7'}
DATABASE TOOL CALLED: Getting price for London
DATABASE TOOL CALLED: Getting price for Paris
{'role': 'system', 'content': "\nYou are a helpful ass